# 실험 제목: 토큰(Token) 계산과 비용 제어

**목표**: LLM이 텍스트를 처리하는 단위인 '토큰'의 개념을 이해하고, `tiktoken`을 사용해 텍스트의 토큰 수를 직접 계산해봅니다. 또한 `max_tokens` 파라미터로 응답 길이를 제어해봅니다.

In [5]:
# 필요한 라이브러리 import
import os
import tiktoken
from dotenv import load_dotenv
from openai import OpenAI

In [6]:
# .env 파일 로드 및 OpenAI 클라이언트 생성
load_dotenv()
client = OpenAI()

### 1. tiktoken으로 토큰 수 계산하기
영어와 한국어의 토큰 소비량이 어떻게 다른지 확인해 봅니다.

In [7]:
# gpt-4o 계열 모델이 사용하는 인코딩 방식 가져오기
encoding = tiktoken.encoding_for_model("gpt-4o-mini")

text_en = "Hello, how are you today?"
text_ko = "안녕하세요, 오늘 하루 어떠신가요?"

tokens_en = encoding.encode(text_en)
tokens_ko = encoding.encode(text_ko)

print(f"영어 문장 글자수: {len(text_en)} -> 토큰 수: {len(tokens_en)}")
print(f"한국어 문장 글자수: {len(text_ko)} -> 토큰 수: {len(tokens_ko)}")
print("\n(일반적으로 영어는 1단어=1토큰에 가깝지만, 한국어는 형태소나 글자 단위로 더 잘게 쪼개져 토큰 소모가 더 큽니다.)")

영어 문장 글자수: 25 -> 토큰 수: 7
한국어 문장 글자수: 19 -> 토큰 수: 11

(일반적으로 영어는 1단어=1토큰에 가깝지만, 한국어는 형태소나 글자 단위로 더 잘게 쪼개져 토큰 소모가 더 큽니다.)


### 2. max_tokens 설정으로 출력 제어하기
`max_tokens`는 모델이 생성할 수 있는 **최대 토큰 수**를 제한합니다. 비용을 관리하거나 응답 길이를 강제로 자를 때 사용합니다.

In [8]:
prompt = "한국의 역사에 대해 설명해줘."

# max_tokens를 20으로 아주 작게 설정
response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": prompt}],
    max_tokens=20  # 생성되는 토큰 수를 최대 20개로 제한
)

output_text = response.choices[0].message.content
finish_reason = response.choices[0].finish_reason

print("[응답 결과]")
print(output_text)
print("\n[종료 사유 (finish_reason)]:", finish_reason)
print("(finish_reason이 'length'이면 max_tokens에 도달하여 문장이 중간에 잘렸음을 의미합니다.)")

[응답 결과]
한국의 역사는 매우 길고 복잡하며 다양한 문화적, 정치적 변화를 겪

[종료 사유 (finish_reason)]: length
(finish_reason이 'length'이면 max_tokens에 도달하여 문장이 중간에 잘렸음을 의미합니다.)


### 3. 잘림 현상 방지하기 (Prompt Engineering)
`max_tokens`를 늘리는 것이 가장 단순한 해결책이지만, 비용 절감을 위해 짧게 답변을 받고 싶다면 **프롬프트 엔지니어링**을 함께 사용해야 합니다.
모델에게 처음부터 "한 문장으로 짧게 요약해줘"라고 지시하면, 잘림 없이 완성된 문장을 받을 확률이 매우 높아집니다.

In [11]:
prompt = "한국의 역사에 대해 설명해줘."

# 시스템 프롬프트로 강제적으로 짧게 쓰도록 지시
response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "system", "content": "당신은 요약의 달인입니다. 반드시 한 문장으로 매우 짧게 답변하세요."},
        {"role": "user", "content": prompt}
    ],
    max_tokens=50  # 여유 있게 조금 늘려주면 안전함
)

output_text = response.choices[0].message.content
finish_reason = response.choices[0].finish_reason

print("[응답 결과]")
print(output_text)
print("\n[종료 사유 (finish_reason)]:", finish_reason)
print("(finish_reason이 'stop'이면 정상적으로 문장을 끝마쳤음을 의미합니다!)")

[응답 결과]
한국은 고대 고조선부터 현대에 이르기까지 다양한 왕조와 사건을 거쳐 발전한 역사적인 국가입니다.

[종료 사유 (finish_reason)]: stop
(finish_reason이 'stop'이면 정상적으로 문장을 끝마쳤음을 의미합니다!)


### 실험 결과 정리

* **토큰 인코딩**: 동일한 의미의 문장이라도 영어보다 한국어 데이터가 토큰을 더 많이 소모한다는 것을 `tiktoken` 실습으로 확인했습니다.
* **max_tokens 제한**: `max_tokens` 파라미터를 작게 설정하면 응답이 강제로 잘리고, `finish_reason`이 `length`로 반환됨을 확인했습니다.
* **Prompt Engineering 결합**: 단순히 토큰 수만 제한하면 문장이 잘리지만, **System Prompt로 짧게 답변하도록 미리 지시**하면 토큰 제한 내에서 문장을 완벽하게 마무리(`finish_reason == 'stop'`)할 수 있습니다.

### Notion 실험 로그 기록용

* **결과**: `tiktoken`을 활용한 한/영 토큰 소비량 비교 및 `max_tokens` 설정을 통한 생성 길이 제한 테스트를 완료함.
* **문제점**: `max_tokens`만 작게 주면 출력 문장이 중간에 잘려버림(`finish_reason: length`).
* **다음 개선 방향**: 토큰 비용을 아끼면서 완전한 문장을 받으려면 System Prompt를 통해 모델에게 명시적으로 출력 길이를 강제하는 방식을 병행해야 함.